# Step 1
 - Extracting Commit Messages for our commit list from the fully cloned repos saved locally

In [1]:
from __future__ import annotations

import csv
import re
import subprocess
from pathlib import Path
from typing import Dict, Set, Optional, Tuple
from collections import defaultdict

# -----------------------------
# Config (edit as needed)
# -----------------------------
CLONE_ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone_Full")

TARGET_COMMITS_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv"
)

# Write output alongside your existing metadata outputs (same folder as PR/Issue JSONs)
OUT_COMMIT_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\CommitMessages_local.csv"
)

# -----------------------------
# Helpers
# -----------------------------

GITHUB_RE = re.compile(r"github\.com[:/]+([^/]+)/([^/]+?)(?:\.git)?$", re.IGNORECASE)

def parse_owner_repo_from_remote(remote_url: str) -> Optional[Tuple[str, str]]:
    """
    Parse owner/repo from typical GitHub remote URLs:
      - https://github.com/owner/repo(.git)
      - git@github.com:owner/repo(.git)
    """
    u = (remote_url or "").strip()
    if not u:
        return None
    m = GITHUB_RE.search(u)
    if not m:
        return None
    return m.group(1), m.group(2)

def load_target_commits(path: Path) -> Dict[str, Set[str]]:
    """
    Load commits per repo from CSV columns:
      - repo_name: expected "owner__repo"
      - commit: sha
    """
    mapping: Dict[str, Set[str]] = defaultdict(set)
    if not path.exists():
        raise FileNotFoundError(f"Target commit CSV not found: {path}")

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            repo_name = (row.get("repo_name") or "").strip()
            sha = (row.get("commit_sha") or "").strip()
            if repo_name and sha:
                mapping[repo_name].add(sha)

    print(f"Loaded target commits for {len(mapping)} repos from: {path}")
    return mapping

def run_git(repo_path: Path, args: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", "-C", str(repo_path), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )


def is_git_repo(path: Path) -> bool:
    """
    Determine if path is a git repo.
    """
    if (path / ".git").exists():
        return True
    p = run_git(path, ["rev-parse", "--is-inside-work-tree"])
    return p.returncode == 0 and p.stdout.strip().lower() == "true"

def get_origin_url(repo_path: Path) -> str:
    """
    Get origin remote URL if available.
    """
    p = run_git(repo_path, ["remote", "get-url", "origin"])
    if p.returncode == 0:
        return p.stdout.strip()
    # fallback: try config key
    p2 = run_git(repo_path, ["config", "--get", "remote.origin.url"])
    return p2.stdout.strip() if p2.returncode == 0 else ""

def index_local_clones(clone_root: Path) -> Dict[str, Path]:
    """
    Build mapping: repo_key ("owner__repo") -> local path.
    Uses origin remote when possible; falls back to folder name if needed.
    """
    if not clone_root.exists():
        raise FileNotFoundError(f"Clone root not found: {clone_root}")

    mapping: Dict[str, Path] = {}

    # Assumption: clones are direct children; if yours are nested, change to rglob("*")
    for child in clone_root.iterdir():
        if not child.is_dir():
            continue
        if not is_git_repo(child):
            continue

        origin = get_origin_url(child)
        parsed = parse_owner_repo_from_remote(origin)
        if parsed:
            owner, repo = parsed
            key = f"{owner}__{repo}"
            mapping[key] = child
        else:
            # fallback to folder name
            mapping[child.name] = child

    print(f"Indexed {len(mapping)} local git repos under: {clone_root}")
    return mapping

def get_commit_message(repo_path: Path, sha: str) -> Tuple[Optional[dict], str]:
    """
    Return (commit_record_dict, error_message).
    commit_record_dict includes subject/body + author/committer metadata.
    """
    # Use US (unit separator) to split fields safely
    sep = "\x1f"
    fmt = sep.join([
        "%H",     # sha
        "%an",    # author name
        "%ae",    # author email
        "%ad",    # author date
        "%cn",    # committer name
        "%ce",    # committer email
        "%cd",    # committer date
        "%s",     # subject
        "%b",     # body (no subject)
    ])

    p = run_git(repo_path, ["show", "-s", f"--date=iso-strict", f"--format={fmt}", sha])
    if p.returncode != 0:
        err = (p.stderr or p.stdout or "").strip()
        return None, err[:2000]

    raw = p.stdout.rstrip("\n")
    parts = raw.split(sep)

    if len(parts) < 9:
        return None, f"Unexpected git output format for {sha}. Raw={raw[:200]}"

    (sha_out, an, ae, ad, cn, ce, cd, subject, body) = parts[:9]

    full_message = subject if not body.strip() else (subject + "\n\n" + body.rstrip())

    rec = {
        "commit_sha": sha_out,
        "author_name": an,
        "author_email": ae,
        "author_date": ad,
        "committer_name": cn,
        "committer_email": ce,
        "committer_date": cd,
        "subject": subject,
        "body": body,
        "full_message": full_message,
    }
    return rec, ""

# -----------------------------
# Main
# -----------------------------
def main() -> None:
    targets = load_target_commits(TARGET_COMMITS_CSV)
    repo_map = index_local_clones(CLONE_ROOT)

    OUT_COMMIT_CSV.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        # PRIMARY KEYS for merge
        "repo_name",       # e.g., owner__repo
        "commit_sha",      # full sha
        # useful metadata
        "subject",
        "body",
        "full_message",
        "author_name",
        "author_email",
        "author_date",
        "committer_name",
        "committer_email",
        "committer_date",
        # traceability
        "local_repo_path",
        "status",
        "error",
    ]

    out_rows = []
    missing_repo = 0
    missing_commit = 0
    ok = 0

    for repo_name, shas in targets.items():
        repo_path = repo_map.get(repo_name)

        if repo_path is None:
            # could not find the repo locally
            for sha in shas:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": sha,
                    "subject": "",
                    "body": "",
                    "full_message": "",
                    "author_name": "",
                    "author_email": "",
                    "author_date": "",
                    "committer_name": "",
                    "committer_email": "",
                    "committer_date": "",
                    "local_repo_path": "",
                    "status": "missing_repo",
                    "error": "Repo not found under CLONE_ROOT (by origin URL or folder name).",
                })
            missing_repo += 1
            continue

        for sha in sorted(shas):
            rec, err = get_commit_message(repo_path, sha)
            if rec is None:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": sha,
                    "subject": "",
                    "body": "",
                    "full_message": "",
                    "author_name": "",
                    "author_email": "",
                    "author_date": "",
                    "committer_name": "",
                    "committer_email": "",
                    "committer_date": "",
                    "local_repo_path": str(repo_path),
                    "status": "missing_commit",
                    "error": err or "Commit not found in local clone (shallow clone or missing history).",
                })
                missing_commit += 1
            else:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": rec["commit_sha"],
                    "subject": rec["subject"],
                    "body": rec["body"],
                    "full_message": rec["full_message"],
                    "author_name": rec["author_name"],
                    "author_email": rec["author_email"],
                    "author_date": rec["author_date"],
                    "committer_name": rec["committer_name"],
                    "committer_email": rec["committer_email"],
                    "committer_date": rec["committer_date"],
                    "local_repo_path": str(repo_path),
                    "status": "ok",
                    "error": "",
                })
                ok += 1

    with OUT_COMMIT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(out_rows)

    print(f"Done. ok={ok}, missing_repo_groups={missing_repo}, missing_commit_rows={missing_commit}")
    print(f"Wrote: {OUT_COMMIT_CSV}")

if __name__ == "__main__":
    main()


Loaded target commits for 431 repos from: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
Indexed 481 local git repos under: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone_Full
Done. ok=532, missing_repo_groups=0, missing_commit_rows=0
Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\CommitMessages_local.csv


# Step 2 - Extracting PR messages as well as the issues

The first step in data process for this RQ starts with reading the commit and PR messages and issues from list of commits detected as instru related commits from the prevoius RQ

In [3]:
from __future__ import annotations
import csv, os, re, time, json
from pathlib import Path
from typing import Optional, List, Dict, Set
from collections import defaultdict

import requests          # pip install requests
from dotenv import load_dotenv  # pip install python-dotenv

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3")
URL_LIST_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\URL_List.csv")
MANIFEST_CSV = WORK_ROOT / "PRs_Issues_manifest.csv"

# Where to store GitHub metadata (PRs + issues)
META_ROOT = WORK_ROOT / "PRs_Issues"

# NEW: path to the commit list produced by your episode analysis
TARGET_COMMITS_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv") 

FETCH_GH_METADATA = True  # set False to dry-run the URL list

# Path to your env file
ENV_FILE = r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"

# === LOAD .env ===
load_dotenv(ENV_FILE)

# Load GitHub tokens: GITHUB_TOKEN_1 ... GITHUB_TOKEN_6
TOKENS = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 4)]
TOKENS = [t for t in TOKENS if t]

if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_Tokens.env (API metadata might be rate limited).")
else:
    print(f"ℹ️ Loaded {len(TOKENS)} GitHub token(s) from All_Tokens.env")
    print("Token lengths:", [len(t) for t in TOKENS])

# index of the current token (0-based)
token_index = 0

GITHUB_API_BASE = "https://api.github.com"

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)

# Regex for issue references like "#123"
ISSUE_REF_RE = re.compile(r"#(\d+)")

# -----------------------------
# Load target commits per repo
# -----------------------------
def load_target_commits(path: Path) -> Dict[str, Set[str]]:
    """
    Load target commit SHAs per repo from obs3_1_change_episodes_commits.csv.

    Uses the 'repo_name' column (e.g., 'connectbot__connectbot') and 'commit'.
    """
    mapping: Dict[str, Set[str]] = defaultdict(set)
    if not path.exists():
        print(f"⚠️ Warning: target commit CSV not found: {path}")
        return mapping

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            repo_name = (row.get("repo_name") or "").strip()  # e.g., "connectbot__connectbot"
            sha = (row.get("commit_sha") or "").strip()
            if repo_name and sha:
                mapping[repo_name].add(sha)

    print(f"ℹ️ Loaded target commits for {len(mapping)} repos from {path}")
    return mapping


TARGET_COMMITS_BY_REPO: Dict[str, Set[str]] = load_target_commits(TARGET_COMMITS_CSV)

# -----------------------------
# Helpers (no git here)
# -----------------------------

def parse_github_owner_repo(url: str) -> Optional[tuple[str, str]]:
    """
    Return (owner, repo) for GitHub URLs, or None if not GitHub.
    Supports https://github.com/owner/repo(.git) and git@github.com:owner/repo(.git).
    """
    u = url.strip()
    # SSH form
    m = re.match(r"^git@github\.com:([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        return m.group(1), m.group(2)

    # HTTPS form
    if "github.com" not in u.lower():
        return None

    # Drop protocol
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u

    # Remove possible query/fragments
    base = base.split("?", 1)[0].split("#", 1)[0]

    parts = [p for p in base.split("/") if p]
    # parts like ["github.com", "owner", "repo(.git)"]
    if len(parts) >= 3 and parts[0].lower().startswith("github.com"):
        owner = parts[1]
        repo = parts[2].removesuffix(".git")
        return owner, repo

    return None


def github_get(path: str, params: Optional[dict] = None) -> list:
    """
    Basic GitHub API GET with pagination.
    Uses round-robin token rotation when hitting rate limits.
    Returns a list of items.
    """
    global token_index

    url = f"{GITHUB_API_BASE}{path}"
    items: list = []
    page = 1

    while True:
        # Build headers with the current token
        headers = {
            # include preview for /commits/{sha}/pulls endpoint as well
            "Accept": "application/vnd.github+json, application/vnd.github.groot-preview+json"
        }
        current_token = TOKENS[token_index] if TOKENS else None
        if current_token:
            headers["Authorization"] = f"Bearer {current_token}"

        q = dict(params or {})
        q.setdefault("per_page", 100)
        q["page"] = page

        resp = requests.get(url, headers=headers, params=q)

        # Detect rate limit
        remaining = resp.headers.get("X-RateLimit-Remaining")
        is_rate_limited = (
            resp.status_code == 403
            and ("rate limit" in resp.text.lower() or remaining == "0")
        )

        if is_rate_limited:
            print(
                f"[rate limit] {path} page={page} with token index {token_index}. "
                f"Remaining={remaining}"
            )
            if TOKENS and len(TOKENS) > 1:
                old_index = token_index
                token_index = (token_index + 1) % len(TOKENS)
                print(f"  -> switching token {old_index} -> {token_index} and retrying...")
                continue
            else:
                print("  -> no alternative tokens; stopping.")
                break

        # If we reach here, it's not a rate-limit error
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break

        if isinstance(data, list):
            items.extend(data)
        else:
            # some endpoints return an object, not a list
            items.append(data)
            break

        if len(data) < q["per_page"]:
            # last page
            break

        page += 1

    return items


# --- Targeted helpers: commit -> PRs, PR -> issues ----

def get_prs_for_commit(owner: str, repo: str, sha: str) -> List[dict]:
    """
    Return list of PRs that include this commit.
    Uses /repos/{owner}/{repo}/commits/{sha}/pulls.
    """
    path = f"/repos/{owner}/{repo}/commits/{sha}/pulls"
    try:
        return github_get(path, params=None)
    except Exception as e:
        print(f"    [warn] commit {sha}: error fetching associated PRs: {e}")
        return []


def fetch_issue_with_comments(owner: str, repo: str, number: int) -> Optional[dict]:
    """
    Fetch a single issue + its comments.
    """
    try:
        issue_list = github_get(f"/repos/{owner}/{repo}/issues/{number}")
        if not issue_list:
            return None
        issue = issue_list[0]
    except Exception as e:
        print(f"    [warn] issue #{number}: error fetching issue: {e}")
        return None

    try:
        comments = github_get(f"/repos/{owner}/{repo}/issues/{number}/comments")
    except Exception as e:
        print(f"    [warn] issue #{number}: error fetching comments: {e}")
        comments = []
    issue["comments"] = comments
    return issue


def extract_issue_numbers_from_pr(pr: dict) -> Set[int]:
    """
    Extract referenced issue numbers from PR body/title/comments/reviews using #123 pattern.
    """
    texts: List[str] = []

    for key in ("title", "body"):
        v = pr.get(key)
        if isinstance(v, str):
            texts.append(v)

    for c in pr.get("issue_comments", []):
        v = c.get("body")
        if isinstance(v, str):
            texts.append(v)

    for c in pr.get("review_comments", []):
        v = c.get("body")
        if isinstance(v, str):
            texts.append(v)

    for r in pr.get("reviews", []):
        v = r.get("body")
        if isinstance(v, str):
            texts.append(v)

    nums: Set[int] = set()
    for t in texts:
        for m in ISSUE_REF_RE.findall(t):
            try:
                nums.add(int(m))
            except ValueError:
                pass
    return nums


def fetch_github_prs_and_issues(owner: str, repo: str, out_dir: Path) -> tuple[int, int]:
    """
    TARGETED VERSION:

    For this (owner, repo), we only fetch metadata for the commits listed
    in obs3_1_change_episodes_commits.csv (per repo).

    For those commits:
      - find associated PRs
      - fetch full PR (details + issue_comments + review_comments + reviews)
      - parse PR text/comments to find #issue references
      - fetch only those issues (+ comments)

    Returns:
        (num_targeted_prs, num_targeted_issues)
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    base_name = f"{owner}__{repo}"

    repo_key = base_name  # matches 'repo_name' in the commits CSV
    commit_shas = TARGET_COMMITS_BY_REPO.get(repo_key, set())

    if not commit_shas:
        print(f"  [meta] No target commits for {owner}/{repo} (repo_name={repo_key}); skipping.")
        # still write empty JSON files for consistency
        with (out_dir / f"{base_name}_PRs.json").open("w", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False, indent=2)
        with (out_dir / f"{base_name}_Issues.json").open("w", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False, indent=2)
        return 0, 0

    print(f"  [meta] Target commits for {owner}/{repo}: {len(commit_shas)}")

    targeted_prs: List[dict] = []
    targeted_issues: List[dict] = []

    seen_pr_numbers: Set[int] = set()
    seen_issue_numbers: Set[int] = set()

    # ---- 1) For each commit, get its PRs and fetch full PR discussion ----
    for sha in sorted(commit_shas):
        prs_for_commit = get_prs_for_commit(owner, repo, sha)
        if not prs_for_commit:
            continue

        for pr_stub in prs_for_commit:
            number = pr_stub.get("number")
            if number is None:
                continue
            if number in seen_pr_numbers:
                # Optionally attach this commit to an existing PR record
                for existing in targeted_prs:
                    if existing.get("number") == number:
                        existing.setdefault("target_commits", [])
                        if sha not in existing["target_commits"]:
                            existing["target_commits"].append(sha)
                continue

            seen_pr_numbers.add(number)

            # PR core details
            try:
                pr_full_list = github_get(f"/repos/{owner}/{repo}/pulls/{number}")
                pr_full = pr_full_list[0] if pr_full_list else pr_stub
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching details: {e}")
                pr_full = pr_stub

            # issue-style comments on the PR
            try:
                issue_comments = github_get(
                    f"/repos/{owner}/{repo}/issues/{number}/comments"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching issue comments: {e}")
                issue_comments = []

            # review comments on specific lines
            try:
                review_comments = github_get(
                    f"/repos/{owner}/{repo}/pulls/{number}/comments"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching review comments: {e}")
                review_comments = []

            # review events (approve/request-changes etc.)
            try:
                reviews = github_get(
                    f"/repos/{owner}/{repo}/pulls/{number}/reviews"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching reviews: {e}")
                reviews = []

            pr_full["issue_comments"] = issue_comments
            pr_full["review_comments"] = review_comments
            pr_full["reviews"] = reviews
            pr_full["target_commits"] = [sha]

            targeted_prs.append(pr_full)

    # ---- 2) From those PRs, find referenced issues and fetch them ----
    for pr in targeted_prs:
        issue_numbers = extract_issue_numbers_from_pr(pr)
        for num in issue_numbers:
            if num in seen_issue_numbers:
                continue
            seen_issue_numbers.add(num)
            issue = fetch_issue_with_comments(owner, repo, num)
            if issue is not None:
                targeted_issues.append(issue)

    # ---- 3) Save as JSON (same filenames as before) ----
    with (out_dir / f"{base_name}_PRs.json").open("w", encoding="utf-8") as f:
        json.dump(targeted_prs, f, ensure_ascii=False, indent=2)

    with (out_dir / f"{base_name}_Issues.json").open("w", encoding="utf-8") as f:
        json.dump(targeted_issues, f, ensure_ascii=False, indent=2)

    return len(targeted_prs), len(targeted_issues)


# -----------------------------
# Main: GH METADATA ONLY (no clone)
# -----------------------------
def main() -> None:
    assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

    rows, ok, fail = [], 0, 0
    with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            url = (row.get("repo_url") or "").strip()
            if not url:
                continue
            t0 = time.time()
            rec = {
                "repo_url": url,
                "status": "unknown",
                "seconds": None,
                "num_prs": None,
                "num_issues": None,
                "error": "",
            }
            try:
                if not FETCH_GH_METADATA:
                    rec["status"] = "skipped"
                    ok += 1
                else:
                    gh = parse_github_owner_repo(url)
                    if gh is None:
                        rec["status"] = "non_github"
                        rec["error"] = "Not a GitHub URL"
                        fail += 1
                    else:
                        owner, repo_name = gh
                        try:
                            num_prs, num_issues = fetch_github_prs_and_issues(
                                owner, repo_name, META_ROOT
                            )
                            rec["status"] = "ok"
                            rec["num_prs"] = num_prs
                            rec["num_issues"] = num_issues
                            ok += 1
                        except Exception as e:
                            msg = f"metadata error: {e}"
                            print(f"  [meta-error] {url}: {msg}")
                            rec["status"] = "error"
                            rec["error"] = msg
                            fail += 1

            except Exception as e:
                rec["status"] = "error"
                rec["error"] = str(e)[:2000]
                fail += 1

            rec["seconds"] = round(time.time() - t0, 2)
            rows.append(rec)
            print(
                f"[{rec['status']}] {url} "
                f"({rec['seconds']}s)  PRs={rec['num_prs']}  Issues={rec['num_issues']}"
            )

    # Write manifest for metadata
    MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["repo_url", "status", "seconds", "num_prs", "num_issues", "error"]
    with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


if __name__ == "__main__":
    main()


ℹ️ Loaded 3 GitHub token(s) from All_Tokens.env
Token lengths: [40, 40, 40]
ℹ️ Loaded target commits for 431 repos from C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
  [meta] Target commits for connectbot/connectbot: 2
[ok] https://github.com/connectbot/connectbot (2.81s)  PRs=2  Issues=0
  [meta] Target commits for ge0rg/aprsdroid: 1
[ok] https://github.com/ge0rg/aprsdroid (1.41s)  PRs=1  Issues=0
  [meta] Target commits for robolectric/robolectric: 2
[ok] https://github.com/robolectric/robolectric (4.31s)  PRs=2  Issues=0
  [meta] Target commits for opendocument-app/OpenDocument.droid: 1
[ok] https://github.com/opendocument-app/OpenDocument.droid (1.4s)  PRs=1  Issues=0
  [meta] Target commits for maxpower47/PinDroid: 1
[ok] https://github.com/maxpower47/PinDroid (1.64s)  PRs=1  Issues=0
  [meta] Target commits for Rajawali/Rajawali: 3
[ok] https://github.com/Rajawali/Rajawali (5.98s)  PRs=3  Issues=3
  

# Step 3: merging the Commit & PR Msgs

In [4]:
from __future__ import annotations

import csv
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional, Set
from collections import defaultdict

# ============================================================
# Config
# ============================================================
BOUNDARY_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv"
)

COMMIT_MSG_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\CommitMessages_local.csv"
)

PRS_ISSUES_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\PRs_Issues"
)

OUT_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\All_Commits_PR_Msg_Iss.csv"
)

# Excel cell max is 32767 chars; keep a safety buffer
MAX_CELL_CHARS = 32500

# Join separator used when concatenating multiple PR/issue messages
SEP = " <SEP> "

ISSUE_REF_RE = re.compile(r"#(\d+)")


# ============================================================
# Safety helpers
# ============================================================
def safe_text(v: Any, max_chars: int = MAX_CELL_CHARS) -> str:
    """
    Ensure a cell is safe:
      - always a string
      - remove NULs
      - normalize newlines
      - hard cap length to max_chars (Excel-safe)
    """
    if v is None:
        s = ""
    elif isinstance(v, str):
        s = v
    else:
        s = str(v)

    # Clean
    s = s.replace("\x00", "")               # remove NULs
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # Truncate (keep within max_chars)
    if len(s) > max_chars:
        tail = "…[TRUNCATED]"
        keep = max_chars - len(tail)
        if keep < 0:
            return tail[:max_chars]
        s = s[:keep] + tail

    return s


def safe_join(parts: List[str], sep: str = SEP, max_chars: int = MAX_CELL_CHARS) -> str:
    """
    Join many strings, truncating the final result to max_chars.
    """
    if not parts:
        return ""
    joined = sep.join([p for p in parts if p])
    return safe_text(joined, max_chars=max_chars)


def norm_repo_key(repo_name: str) -> str:
    return (repo_name or "").strip()


def norm_sha(sha: str) -> str:
    return (sha or "").strip()


# ============================================================
# Loaders
# ============================================================
def read_csv_rows(path: Path) -> Tuple[List[dict], List[str]]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", newline="", encoding="utf-8-sig") as f:
        r = csv.DictReader(f)
        rows = list(r)
        headers = r.fieldnames or []
    return rows, headers


def load_commit_messages(path: Path) -> Dict[Tuple[str, str], dict]:
    """
    Map (repo_name, commit_sha) -> commit record
    """
    rows, _ = read_csv_rows(path)

    m: Dict[Tuple[str, str], dict] = {}
    for row in rows:
        repo = norm_repo_key(row.get("repo_name", ""))
        sha = norm_sha(row.get("commit_sha", "") or row.get("commit", ""))
        if repo and sha:
            m[(repo, sha)] = row
    print(f"Loaded commit messages: {len(m)} rows from {path}")
    return m


def load_repo_prs_issues(prs_issues_dir: Path) -> Tuple[
    Dict[str, List[dict]],
    Dict[str, Dict[int, dict]],
    Dict[Tuple[str, str], List[dict]],
]:
    """
    Returns:
      repo_to_prs: repo_key -> list of PR dicts
      repo_to_issues_by_num: repo_key -> {issue_number: issue_dict}
      pr_by_repo_commit: (repo_key, sha) -> list of PR dicts
    """
    repo_to_prs: Dict[str, List[dict]] = defaultdict(list)
    repo_to_issues_by_num: Dict[str, Dict[int, dict]] = defaultdict(dict)
    pr_by_repo_commit: Dict[Tuple[str, str], List[dict]] = defaultdict(list)

    if not prs_issues_dir.exists():
        raise FileNotFoundError(prs_issues_dir)

    pr_files = sorted(prs_issues_dir.glob("*_PRs.json"))
    issue_files = sorted(prs_issues_dir.glob("*_Issues.json"))

    # --- PRs ---
    for fp in pr_files:
        base = fp.name[:-len("_PRs.json")]  # owner__repo
        repo_key = base
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
            if not isinstance(data, list):
                data = [data]
        except Exception as e:
            print(f"[warn] Could not read PR file {fp}: {e}")
            continue

        repo_to_prs[repo_key].extend(data)

        for pr in data:
            commits = pr.get("target_commits") or []
            if isinstance(commits, str):
                commits = [commits]
            for sha in commits:
                sha = norm_sha(sha)
                if sha:
                    pr_by_repo_commit[(repo_key, sha)].append(pr)

    # --- Issues ---
    for fp in issue_files:
        base = fp.name[:-len("_Issues.json")]
        repo_key = base
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
            if not isinstance(data, list):
                data = [data]
        except Exception as e:
            print(f"[warn] Could not read Issue file {fp}: {e}")
            continue

        for iss in data:
            num = iss.get("number")
            try:
                if num is not None:
                    repo_to_issues_by_num[repo_key][int(num)] = iss
            except Exception:
                pass

    print(f"Loaded PR repos: {len(repo_to_prs)}  | Issue repos: {len(repo_to_issues_by_num)}")
    return repo_to_prs, repo_to_issues_by_num, pr_by_repo_commit


# ============================================================
# Extractors / formatters
# ============================================================
def extract_issue_numbers_from_pr(pr: dict) -> Set[int]:
    texts: List[str] = []

    for key in ("title", "body"):
        v = pr.get(key)
        if isinstance(v, str) and v:
            texts.append(v)

    for c in pr.get("issue_comments", []) or []:
        v = c.get("body")
        if isinstance(v, str) and v:
            texts.append(v)

    for c in pr.get("review_comments", []) or []:
        v = c.get("body")
        if isinstance(v, str) and v:
            texts.append(v)

    for r in pr.get("reviews", []) or []:
        v = r.get("body")
        if isinstance(v, str) and v:
            texts.append(v)

    nums: Set[int] = set()
    for t in texts:
        for m in ISSUE_REF_RE.findall(t):
            try:
                nums.add(int(m))
            except Exception:
                pass
    return nums


def pr_url(pr: dict) -> str:
    return safe_text(pr.get("html_url") or pr.get("url") or "")


def issue_url(issue: dict) -> str:
    return safe_text(issue.get("html_url") or issue.get("url") or "")


def collect_pr_fields(prs: List[dict]) -> dict:
    nums, titles, bodies, urls = [], [], [], []
    comments_and_reviews = []

    for pr in prs:
        n = pr.get("number")
        if n is not None:
            nums.append(str(n))

        titles.append(safe_text(pr.get("title") or ""))
        bodies.append(safe_text(pr.get("body") or ""))
        urls.append(pr_url(pr))

        # Comments (issue-style) + review comments + reviews
        blocks: List[str] = []
        for c in pr.get("issue_comments", []) or []:
            b = c.get("body")
            if isinstance(b, str) and b.strip():
                blocks.append(b.strip())
        for c in pr.get("review_comments", []) or []:
            b = c.get("body")
            if isinstance(b, str) and b.strip():
                blocks.append(b.strip())
        for r in pr.get("reviews", []) or []:
            b = r.get("body")
            if isinstance(b, str) and b.strip():
                blocks.append(b.strip())

        if blocks:
            comments_and_reviews.append(safe_join([safe_text(x) for x in blocks], sep=SEP))

    return {
        "pr_count": str(len(prs)),
        "pr_numbers": safe_join(nums),
        "pr_titles": safe_join(titles),
        "pr_bodies": safe_join(bodies),
        "pr_comments_and_reviews": safe_join(comments_and_reviews),
        "pr_urls": safe_join(urls),
    }


def collect_issue_fields(issues: List[dict]) -> dict:
    nums, titles, bodies, urls = [], [], [], []
    comments_all = []

    for iss in issues:
        n = iss.get("number")
        if n is not None:
            nums.append(str(n))

        titles.append(safe_text(iss.get("title") or ""))
        bodies.append(safe_text(iss.get("body") or ""))
        urls.append(issue_url(iss))

        blocks: List[str] = []
        for c in iss.get("comments", []) or []:
            b = c.get("body")
            if isinstance(b, str) and b.strip():
                blocks.append(b.strip())
        if blocks:
            comments_all.append(safe_join([safe_text(x) for x in blocks], sep=SEP))

    return {
        "issue_count": str(len(issues)),
        "issue_numbers": safe_join(nums),
        "issue_titles": safe_join(titles),
        "issue_bodies": safe_join(bodies),
        "issue_comments": safe_join(comments_all),
        "issue_urls": safe_join(urls),
    }


# ============================================================
# Main merge
# ============================================================
def main() -> None:
    boundary_rows, boundary_headers = read_csv_rows(BOUNDARY_CSV)

    commit_map = load_commit_messages(COMMIT_MSG_CSV)
    _, issues_by_repo_num, prs_by_repo_commit = load_repo_prs_issues(PRS_ISSUES_DIR)

    # Output headers = boundary + commit fields + PR/Issue aggregate fields
    commit_fields = [
        "repo_full_name",
        "commit_subject",
        "commit_body",
        "commit_full_message",
        "author_name",
        "author_email",
        "author_date",
        "committer_name",
        "committer_email",
        "committer_date",
    ]
    pr_issue_fields = [
        "pr_count",
        "pr_numbers",
        "pr_titles",
        "pr_bodies",
        "pr_comments_and_reviews",
        "pr_urls",
        "issue_count",
        "issue_numbers",
        "issue_titles",
        "issue_bodies",
        "issue_comments",
        "issue_urls",
    ]

    # preserve boundary order, append new columns not already present
    out_headers = list(boundary_headers)
    for h in commit_fields + pr_issue_fields:
        if h not in out_headers:
            out_headers.append(h)

    out_rows: List[dict] = []

    missing_commit_msg = 0
    for row in boundary_rows:
        repo_name = norm_repo_key(row.get("repo_name", ""))
        sha = norm_sha(row.get("commit_sha", "") or row.get("commit", "") or row.get("commit_id", ""))

        # Start from boundary row (copy, but sanitize)
        out = {k: safe_text(row.get(k, "")) for k in out_headers}

        # --- Commit messages ---
        cm = commit_map.get((repo_name, sha))
        if not cm:
            missing_commit_msg += 1
            cm = {}

        # Fill commit fields (Excel-safe length cap)
        out["repo_full_name"] = safe_text(
            cm.get("repo_full_name") or row.get("full_name") or row.get("repo_full_name") or ""
        )
        out["commit_subject"] = safe_text(cm.get("subject") or cm.get("commit_subject") or "")
        out["commit_body"] = safe_text(cm.get("body") or cm.get("commit_body") or "")
        out["commit_full_message"] = safe_text(cm.get("full_message") or cm.get("commit_full_message") or "")

        out["author_name"] = safe_text(cm.get("author_name") or "")
        out["author_email"] = safe_text(cm.get("author_email") or "")
        out["author_date"] = safe_text(cm.get("author_date") or "")
        out["committer_name"] = safe_text(cm.get("committer_name") or "")
        out["committer_email"] = safe_text(cm.get("committer_email") or "")
        out["committer_date"] = safe_text(cm.get("committer_date") or "")

        # --- PRs for this commit ---
        prs = prs_by_repo_commit.get((repo_name, sha), [])
        pr_pack = collect_pr_fields(prs)
        out.update({k: safe_text(v) for k, v in pr_pack.items()})

        # --- Issues referenced by those PRs (only if present in Issues.json) ---
        issue_nums: Set[int] = set()
        for pr in prs:
            issue_nums |= extract_issue_numbers_from_pr(pr)

        issues: List[dict] = []
        repo_issues = issues_by_repo_num.get(repo_name, {})
        for n in sorted(issue_nums):
            iss = repo_issues.get(n)
            if iss:
                issues.append(iss)

        issue_pack = collect_issue_fields(issues)
        out.update({k: safe_text(v) for k, v in issue_pack.items()})

        out_rows.append(out)

    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

    # IMPORTANT: quote everything so commas/newlines don’t break parsing;
    # utf-8-sig makes Excel open it cleanly.
    with OUT_CSV.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=out_headers,
            delimiter=",",
            quoting=csv.QUOTE_ALL,
            lineterminator="\n",
            extrasaction="ignore",
        )
        w.writeheader()
        for r in out_rows:
            # Final safety pass (cap every cell)
            w.writerow({k: safe_text(r.get(k, "")) for k in out_headers})

    print(f"Done. Wrote: {OUT_CSV}")
    print(f"Rows: {len(out_rows)} | boundary rows missing commit message match: {missing_commit_msg}")


if __name__ == "__main__":
    main()


Loaded commit messages: 532 rows from C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\CommitMessages_local.csv
Loaded PR repos: 481  | Issue repos: 79
Done. Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\All_Commits_PR_Msg_Iss.csv
Rows: 599 | boundary rows missing commit message match: 0


# Step 3 - Creating the Episod level file
it includes episode rows and for each one it has start commit and end commit including all the messages ready for grouping

In [5]:
import os
import csv
import pandas as pd

# -----------------------------
# Settings
# -----------------------------
CELL_CHAR_LIMIT = 32500
SEP_TOKEN = "<SEP>"

COMMITS_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\All_Commits_PR_Msg_Iss.csv"
EPISODES_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Change_Episodes.csv"
OUTPUT_CSV  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\All_episodes_with_messages.csv"


# -----------------------------
# Helpers
# -----------------------------
def _ensure_exists(path: str) -> None:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

def _to_str(x) -> str:
    if x is None:
        return ""
    s = str(x)
    return "" if s.lower() == "nan" else s

def _truncate(s: str, limit: int = CELL_CHAR_LIMIT) -> str:
    s = _to_str(s)
    if len(s) <= limit:
        return s
    suffix = f"...[TRUNCATED {len(s) - limit} chars]"
    cut = max(0, limit - len(suffix))
    return s[:cut] + suffix

def _split_sep(s: str) -> list[str]:
    s = _to_str(s)
    if not s.strip():
        return []
    # split on SEP_TOKEN whether surrounded by spaces or not
    parts = [p.strip() for p in s.split(SEP_TOKEN)]
    return [p for p in parts if p]

def _join_nonempty(parts: list[str], joiner: str = " || ") -> str:
    parts = [_to_str(p).strip() for p in parts if _to_str(p).strip()]
    return joiner.join(parts)

def _issue_summary(issue_numbers: str, issue_titles: str) -> str:
    nums = _split_sep(issue_numbers)
    titles = _split_sep(issue_titles)

    # If stored without SEP for singletons, keep as-is
    if not nums and _to_str(issue_numbers).strip():
        nums = [_to_str(issue_numbers).strip()]
    if not titles and _to_str(issue_titles).strip():
        titles = [_to_str(issue_titles).strip()]

    n = max(len(nums), len(titles))
    out = []
    for i in range(n):
        num = nums[i] if i < len(nums) else ""
        title = titles[i] if i < len(titles) else ""
        if num and title:
            out.append(f"#{num}: {title}")
        elif num:
            out.append(f"#{num}")
        elif title:
            out.append(title)

    return _join_nonempty(out, " || ")


def _build_index(df: pd.DataFrame, flag_col: str) -> dict:
    """
    Build an index keyed by (repo_name, episode_index, commit_sha) -> row(dict),
    using rows where df[flag_col] == '1'.
    """
    idx = {}
    if flag_col not in df.columns:
        return idx
    sub = df[df[flag_col].astype(str) == "1"]
    for r in sub.to_dict("records"):
        key = (_to_str(r.get("repo_name")), _to_str(r.get("episode_index")), _to_str(r.get("commit_sha")))
        if key not in idx:
            idx[key] = r
    return idx

def _build_any_index(df: pd.DataFrame) -> dict:
    """
    Fallback index keyed by (repo_name, commit_sha) -> first row(dict)
    """
    idx = {}
    for r in df.to_dict("records"):
        key = (_to_str(r.get("repo_name")), _to_str(r.get("commit_sha")))
        if key not in idx:
            idx[key] = r
    return idx


# -----------------------------
# Main
# -----------------------------
def main():
    _ensure_exists(COMMITS_CSV)
    _ensure_exists(EPISODES_CSV)

    commits = pd.read_csv(COMMITS_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")
    episodes = pd.read_csv(EPISODES_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    # Build indices for precise start/end boundary lookup
    start_idx = _build_index(commits, "is_episode_start_boundary")
    end_idx   = _build_index(commits, "is_episode_end_boundary")
    any_idx   = _build_any_index(commits)

    out_rows = []

    for e in episodes.to_dict("records"):
        repo_name = _to_str(e.get("repo_name"))
        full_name = repo_name.replace("__", ".")  # match your sample output
        epi_idx = _to_str(e.get("episode_index"))

        start_sha = _to_str(e.get("episode_start_commit_sha"))
        end_sha   = _to_str(e.get("episode_end_commit_sha"))

        # Prefer boundary-flag rows; fallback to any row for that commit
        start_row = start_idx.get((repo_name, epi_idx, start_sha)) or any_idx.get((repo_name, start_sha), {})
        end_row = {}
        if end_sha.strip():
            end_row = end_idx.get((repo_name, epi_idx, end_sha)) or any_idx.get((repo_name, end_sha), {})

        # Boundary metadata (from commit rows)
        env_styles = _join_nonempty([start_row.get("env_style", ""), end_row.get("env_style", "")], " || ") if end_row else _to_str(start_row.get("env_style", ""))
        boundary_event_types = _join_nonempty([start_row.get("event_type", ""), end_row.get("event_type", "")], " || ") if end_row else _to_str(start_row.get("event_type", ""))
        # Prefer UTC date if present; else raw date
        start_dt = _to_str(start_row.get("event_date_utc")) or _to_str(start_row.get("event_date_raw"))
        end_dt = _to_str(end_row.get("event_date_utc")) or _to_str(end_row.get("event_date_raw"))
        boundary_event_dates = _join_nonempty([start_dt, end_dt], " || ") if end_row else start_dt

        row_out = {
            # columns matching your sample file
            "repo_name": repo_name,
            "full_name": full_name,

            "episode_index": epi_idx,
            "episode_start_utc": _to_str(e.get("episode_start_utc")),
            "episode_end_utc": _to_str(e.get("episode_end_utc")),
            "episode_duration_days": _to_str(e.get("episode_duration_days")),

            "env_styles": env_styles,
            "boundary_event_types": boundary_event_types,
            "boundary_event_dates": boundary_event_dates,

            "episode_start_commit_sha": start_sha,
            "episode_end_commit_sha": end_sha,

            "start_commit_subject": _to_str(start_row.get("commit_subject")),
            "start_commit_body": _to_str(start_row.get("commit_body")),
            "start_pr_title": _to_str(start_row.get("pr_titles")),
            "start_pr_body": _to_str(start_row.get("pr_bodies")),
            "start_issue_summary": _issue_summary(start_row.get("issue_numbers", ""), start_row.get("issue_titles", "")),

            "end_commit_subject": _to_str(end_row.get("commit_subject")) if end_row else "",
            "end_commit_body": _to_str(end_row.get("commit_body")) if end_row else "",
            "end_pr_title": _to_str(end_row.get("pr_titles")) if end_row else "",
            "end_pr_body": _to_str(end_row.get("pr_bodies")) if end_row else "",
            "end_issue_summary": _issue_summary(end_row.get("issue_numbers", ""), end_row.get("issue_titles", "")) if end_row else "",
        }

        out_rows.append(row_out)

    # Enforce exact column order (same as your sample)
    out_cols = [
        "repo_name","full_name","episode_index","episode_start_utc","episode_end_utc","episode_duration_days",
        "env_styles","boundary_event_types","boundary_event_dates",
        "episode_start_commit_sha","episode_end_commit_sha",
        "start_commit_subject","start_commit_body","start_pr_title","start_pr_body","start_issue_summary",
        "end_commit_subject","end_commit_body","end_pr_title","end_pr_body","end_issue_summary"
    ]

    out_df = pd.DataFrame(out_rows, columns=out_cols)

    # Apply safe per-cell truncation (prevents Excel cell overflow / messy CSV viewing)
    for c in out_df.columns:
        out_df[c] = out_df[c].map(lambda x: _truncate(x, CELL_CHAR_LIMIT))

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

    # Write CSV with strong quoting to prevent "spilling" into other columns
    out_df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        lineterminator="\n"
    )

    print(f"Saved: {OUTPUT_CSV}")
    print(f"Rows: {len(out_df):,} | Cols: {len(out_df.columns)}")


if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\All_episodes_with_messages.csv
Rows: 521 | Cols: 21


Step 4 - detecting the blank messages lines to exclude them from subcat labeling process

In [12]:
import os
import pandas as pd

# ---- paths ----
input_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\All_episodes_with_messages.csv"
output_dir = os.path.dirname(input_path)
output_path = os.path.join(output_dir, "All_episodes_with_messages_TrueFalse.csv")

# ---- read ----
df = pd.read_csv(input_path, dtype=str, keep_default_na=False)  # keep_default_na=False keeps blanks as ""

# ---- columns ----
start_cols = [
    "start_commit_subject",
    "start_commit_body",
    "start_pr_title",
    "start_pr_body",
    "start_issue_summary",
]
end_cols = [
    "end_commit_subject",
    "end_commit_body",
    "end_pr_title",
    "end_pr_body",
    "end_issue_summary",
]

missing = [c for c in (start_cols + end_cols) if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# ---- compute flags (True if any field has non-blank text after stripping) ----
df["Start_message"] = (
    df[start_cols]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.strip())
    .ne("")
    .any(axis=1)
)

df["End_message"] = (
    df[end_cols]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.strip())
    .ne("")
    .any(axis=1)
)

# ---- save ----
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Saved:", output_path)


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\All_episodes_with_messages_TrueFalse.csv


## Step 4: Intention labeling by keywords

In [2]:
from __future__ import annotations

import re
from pathlib import Path
import pandas as pd

# -----------------------------
# Config: paths & column names
# -----------------------------
INPUT_PATH = Path(r"D:\3 - RQ3_2\Intention_Detection\All_episodes_with_messages.csv")
OUTPUT_PATH = Path(r"D:\3 - RQ3_2\Intention_Detection\All_episodes_with_messages_with_intentions_keywords.csv")

START_COLS = [
    "start_commit_subject",
    "start_commit_body",
    "start_pr_title",
    "start_pr_body",
    "start_issue_summary",
]

END_COLS = [
    "end_commit_subject",
    "end_commit_body",
    "end_pr_title",
    "end_pr_body",
    "end_issue_summary",
]

END_COMMIT_COL = "episode_end_commit_sha"

# -----------------------------
# Keyword dictionaries
# -----------------------------
ci_keywords = [
    "ci", "github actions", "action", "actions", "workflow", "workflows",
    "travis", "circleci", "bitrise", "jenkins", "gitlab ci", "buildkite",
    "azure pipelines", "pipeline", "runner", "job", "jobs",
]

test_keywords = [
    "test", "tests", "testing", "unittest", "unit test", "unit tests",
    "androidtest", "connectedandroidtest", "instrumentation", "instrumentation test",
    "instrumentation tests", "ui test", "ui tests", "uitest", "espresso",
    "robolectric", "detox", "snapshot test", "screenshot test", "coverage",
    "jacoco", "lint test", "integration test", "integration tests",
    "e2e", "end-to-end", "end to end", "functional test", "acceptance test",
]

migrate_keywords = [
    "migrate", "migration", "move to", "moving to", "switch to", "switch from",
    "replace", "replaced", "rewrite", "rewrote", "port", "ported",
    "deprecate", "deprecated", "retire", "retired", "drop travis",
    "drop circleci", "drop bitrise", "new ci", "ci v2",
]

scope_keywords = [
    "e2e", "end-to-end", "end to end",
    "integration test", "integration tests",
    "instrumentation test", "instrumentation tests",
    "ui test", "ui tests", "uitest",
    "espresso", "detox", "firebase test lab", "ftl",
    "device farm", "device lab", "browserstack", "saucelabs", "sauce labs",
    "matrix", "multi-device", "multi device", "multiple devices",
    "gradle managed device", "managed device", "gmd", "benchmark",
    "screenshot test", "snapshot test", "golden test",
]

release_keywords = [
    "release", "releasing", "deploy", "deployment", "deploying",
    "publish", "publishing", "uploaded to", "upload to",
    "play store", "google play", "app store", "testflight",
    "beta", "production", "release pipeline", "release workflow",
    "version bump", "bump version", "bump to", "tag", "tagged",
    "artifact", "maven central", "jitpack", "fastlane",
    "signed apk", "bundle", "aab", "apk", "rollout", "roll out",
]

cleanup_keywords = [
    "clean up", "cleanup", "tidy", "tidied", "remove", "removed", "drop", "dropped",
    "delete", "deleted", "simplify", "simplified", "refactor", "refactored",
    "restructure", "restructured", "re-organize", "reorganize", "consolidate",
    "deduplicate", "dedup", "dedupe", "unused", "legacy", "obsolete",
    "dead code", "strip", "trim", "split workflow", "combine workflow",
    "rename workflow", "rename job", "cleanup ci", "clean ci",
]

env_keywords = [
    "emulator", "emulators", "emu", "device", "devices", "avd",
    "virtual device", "gmd", "managed device", "test matrix",
]

perf_keywords = [
    "flaky", "flakiness", "flake", "deflake", "unstable", "stability",
    "stable", "more stable", "less stable", "less flaky",
    "fixed flaky", "slow", "slowness", "slowdown", "faster",
    "speed up", "speedup", "performance", "perf", "timeout",
    "time-out", "time out", "hang", "hanging", "hangs",
    "crash", "crashes", "crashing", "intermittent", "intermittently",
    "non-deterministic", "non deterministic", "sporadic",
    "retry", "retries", "rerun", "re-run", "stuck", "stalls", "stalled",
]

# -----------------------------
# Matching helpers (regex-based)
# -----------------------------
def compile_phrase_pattern(phrases: list[str], word_boundary_for_short: bool = True) -> re.Pattern:
    """
    Build a regex that matches any phrase. Uses word boundaries for very short tokens to reduce false positives.
    """
    alts = []
    for p in phrases:
        p = p.strip().lower()
        if not p:
            continue
        escaped = re.escape(p)
        if word_boundary_for_short and len(p) <= 3 and " " not in p:
            alts.append(rf"\b{escaped}\b")
        else:
            alts.append(escaped)
    return re.compile(r"(?:%s)" % "|".join(alts), re.IGNORECASE)

CI_RE = compile_phrase_pattern(ci_keywords)
TEST_RE = compile_phrase_pattern(test_keywords)
MIGRATE_RE = compile_phrase_pattern(migrate_keywords)
SCOPE_RE = compile_phrase_pattern(scope_keywords)
RELEASE_RE = compile_phrase_pattern(release_keywords)
CLEANUP_RE = compile_phrase_pattern(cleanup_keywords)
ENV_RE = compile_phrase_pattern(env_keywords)
PERF_RE = compile_phrase_pattern(perf_keywords)

def detect_intentions_from_text(text: str | None) -> str | None:
    if not isinstance(text, str):
        return None
    t = text.strip()
    if not t:
        return None

    labels: set[str] = set()

    if CI_RE.search(t) and TEST_RE.search(t):
        labels.add("Introduce / strengthen CI-backed tests")

    if MIGRATE_RE.search(t) and CI_RE.search(t):
        labels.add("Migrate or modernise CI infrastructure")

    if SCOPE_RE.search(t):
        labels.add("Expand test scope or capabilities")

    if RELEASE_RE.search(t):
        labels.add("Automate or integrate release workflows")

    if CLEANUP_RE.search(t) and (CI_RE.search(t) or ENV_RE.search(t)):
        labels.add("Clean up or simplify CI / environment configuration")

    if PERF_RE.search(t):
        labels.add("Address performance or stability issues")

    return ", ".join(sorted(labels)) if labels else None

def build_text(row: pd.Series, cols: list[str]) -> str:
    parts: list[str] = []
    for c in cols:
        if c in row:
            val = row[c]
            if isinstance(val, str) and val.strip():
                parts.append(val.strip())
    return "\n".join(parts)

def main() -> None:
    df = pd.read_csv(INPUT_PATH)

    df["start_intention"] = df.apply(
        lambda r: detect_intentions_from_text(build_text(r, START_COLS)),
        axis=1,
    )

    def compute_end_intention(row: pd.Series) -> str | None:
        if pd.isna(row.get(END_COMMIT_COL)):
            return None
        return detect_intentions_from_text(build_text(row, END_COLS))

    df["end_intention"] = df.apply(compute_end_intention, axis=1)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated file with intentions to: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()


Saved updated file with intentions to: D:\3 - RQ3_2\Intention_Detection\All_episodes_with_messages_with_intentions_keywords.csv
